# 07 - Predicción de Plagas del Día de Hoy
## Sistema de Predicción Temprana de Plagas - Sierra del Patlachique

**Objetivo:** Mostrar predicción de riesgo de plagas para hoy

**Responsabilidades:**
- ✓ Cargar datos climáticos del día
- ✓ Cargar modelo entrenado (XGBoost)
- ✓ Hacer predicción para hoy
- ✓ Mostrar gráficas interactivas
- ✓ Alertar si hay riesgo

**Entrada:** Datos climáticos actuales + Modelo entrenado

**Salida:** Predicciones con gráficas interactivas

In [1]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pickle
import os
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

print("✓ Librerías cargadas correctamente")

# Cargar datos
df = pd.read_csv("../data/features/datos_patlachique_features.csv", 
                  index_col='fecha', parse_dates=True)

print(f"✓ Datos cargados: últimas {len(df)} fechas")
print(f"  Última fecha: {df.index.max().date()}")

✓ Librerías cargadas correctamente
✓ Datos cargados: últimas 817 fechas
  Última fecha: 2026-03-27


## 1. CARGAR MODELO Y DATOS

**Proceso:**
1. Cargar modelo XGBoost entrenado
2. Cargar scaler de normalización
3. Obtener últimas variables del día
4. Hacer predicción

In [2]:
print("\n" + "="*70)
print("1. CARGAR MODELO Y SCALER")
print("="*70)

# Cargar modelo entrenado
with open("../models/modelo_chapulin_XGBoost.pkl", 'rb') as f:
    modelo = pickle.load(f)

# Cargar scaler
with open("../models/scaler.pkl", 'rb') as f:
    scaler = pickle.load(f)

# Cargar lista de features
with open("../models/features_list.pkl", 'rb') as f:
    features = pickle.load(f)

print(f"✓ Modelo cargado: XGBoost")
print(f"✓ Features: {len(features)}")
print(f"✓ Scaler cargado")

# Obtener últimas variables del día
ultima_fecha = df.index[-1]
X_hoy = df[features].iloc[-1:].values

# Normalizar
X_hoy_scaled = scaler.transform(X_hoy)

# Hacer predicción
prediccion = modelo.predict(X_hoy_scaled)[0]
probabilidad = modelo.predict_proba(X_hoy_scaled)[0][1]

print(f"\n📅 PREDICCIÓN PARA: {ultima_fecha.strftime('%Y-%m-%d')}")
print(f"   Predicción: {'🔴 RIESGO' if prediccion == 1 else '🟢 SIN RIESGO'}")
print(f"   Probabilidad: {probabilidad*100:.1f}%")


1. CARGAR MODELO Y SCALER
✓ Modelo cargado: XGBoost
✓ Features: 15
✓ Scaler cargado

📅 PREDICCIÓN PARA: 2026-03-27
   Predicción: 🟢 SIN RIESGO
   Probabilidad: 0.0%


## 2. VARIABLES CLIMÁTICAS DE HOY

**Muestra:** Temperatura, lluvia, humedad, etc.

In [3]:
print("\n" + "="*70)
print("2. VARIABLES CLIMÁTICAS DE HOY")
print("="*70)

# Obtener valores de hoy
hoy_completo = df.iloc[-1]

print(f"\n🌡️  TEMPERATURA:")
print(f"   Media: {hoy_completo['temp_media_C']:.1f}°C")

print(f"\n💧 LLUVIA:")
print(f"   Hoy: {hoy_completo['lluvia_mm']:.1f} mm")
print(f"   Últimos 7 días: {hoy_completo['lluvia_7d']:.1f} mm")
print(f"   Últimos 14 días: {hoy_completo['lluvia_14d']:.1f} mm")

print(f"\n💨 HUMEDAD:")
print(f"   Relativa: {hoy_completo['humedad_relativa_pct']:.1f}%")
print(f"   Suelo: {hoy_completo['humedad_suelo_frac']:.2f}")

print(f"\n📊 INDICADORES:")
print(f"   Días secos (14d): {hoy_completo['dias_secos_14d']:.0f}")
print(f"   Días HR<70% (14d): {hoy_completo['dias_hr_baja_14d']:.0f}")


2. VARIABLES CLIMÁTICAS DE HOY

🌡️  TEMPERATURA:
   Media: 13.1°C

💧 LLUVIA:
   Hoy: 0.5 mm
   Últimos 7 días: 4.9 mm
   Últimos 14 días: 26.7 mm

💨 HUMEDAD:
   Relativa: 65.4%
   Suelo: 0.48

📊 INDICADORES:
   Días secos (14d): 1
   Días HR<70% (14d): 13


## 3. GRÁFICAS INTERACTIVAS

**Visualización 1:** Gauge (indicador de riesgo)

In [4]:
print("\n" + "="*70)
print("3. GRÁFICAS INTERACTIVAS")
print("="*70)

# GRÁFICA 1: GAUGE (Indicador de riesgo)
fig1 = go.Figure(data=[go.Indicator(
    mode="gauge+number+delta",
    value=probabilidad * 100,
    title={'text': "Probabilidad de Riesgo - Chapulín"},
    delta={'reference': 50},
    gauge={
        'axis': {'range': [0, 100]},
        'bar': {'color': '#E74C3C'},
        'steps': [
            {'range': [0, 25], 'color': '#27AE60'},      # Verde
            {'range': [25, 50], 'color': '#F39C12'},     # Naranja
            {'range': [50, 75], 'color': '#E67E22'},     # Naranja oscuro
            {'range': [75, 100], 'color': '#C0392B'}     # Rojo
        ],
        'threshold': {
            'line': {'color': 'red', 'width': 4},
            'thickness': 0.75,
            'value': 70
        }
    }
)])

fig1.update_layout(
    height=500,
    width=600,
    template='plotly_white'
)

fig1.show()
os.makedirs("../reports", exist_ok=True)
fig1.write_html("../reports/07_gauge_riesgo_hoy.html")
print("✓ Gráfica 1 guardada: 07_gauge_riesgo_hoy.html")


3. GRÁFICAS INTERACTIVAS


✓ Gráfica 1 guardada: 07_gauge_riesgo_hoy.html


## 4. TARJETA DE ALERTA

**Muestra:** Estado actual y recomendación

In [5]:
# GRÁFICA 2: TARJETA DE ALERTA
if probabilidad >= 0.7:
    alerta = "🔴 ALERTA ROJA"
    color_alerta = '#C0392B'
    recomendacion = "Riesgo ALTO - Se recomienda intervención inmediata"
elif probabilidad >= 0.5:
    alerta = "🟠 ALERTA NARANJA"
    color_alerta = '#E67E22'
    recomendacion = "Riesgo MODERADO - Monitorear de cerca"
elif probabilidad >= 0.3:
    alerta = "🟡 ALERTA AMARILLA"
    color_alerta = '#F39C12'
    recomendacion = "Riesgo BAJO - Mantener vigilancia"
else:
    alerta = "🟢 SIN ALERTA"
    color_alerta = '#27AE60'
    recomendacion = "Sin riesgo detectado"

fig2 = go.Figure()

# Fondo de alerta
fig2.add_shape(type="rect", x0=0, y0=0, x1=1, y1=1,
              fillcolor=color_alerta, opacity=0.1, line_width=0)

# Texto
fig2.add_annotation(
    text=f"<b>{alerta}</b><br>" +
         f"Probabilidad: {probabilidad*100:.1f}%<br>" +
         f"<br>{recomendacion}",
    xref="paper", yref="paper",
    x=0.5, y=0.5,
    showarrow=False,
    font=dict(size=16, color=color_alerta, family="Arial Black"),
    align="center"
)

fig2.update_layout(
    title=f"Estado Actual - {ultima_fecha.strftime('%d de %B, %Y')}",
    height=300,
    width=800,
    template='plotly_white',
    xaxis=dict(visible=False),
    yaxis=dict(visible=False)
)

fig2.show()
fig2.write_html("../reports/07_alerta_hoy.html")
print("✓ Gráfica 2 guardada: 07_alerta_hoy.html")

✓ Gráfica 2 guardada: 07_alerta_hoy.html


## 5. VARIABLES CLIMÁTICAS - INDICADORES

**Muestra:** Temperatura, lluvia, humedad, humedad suelo en 4 medidores

In [6]:
# GRÁFICA 3: VARIABLES CLIMÁTICAS
fig3 = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        f"Temperatura: {hoy_completo['temp_media_C']:.1f}°C",
        f"Lluvia: {hoy_completo['lluvia_mm']:.1f} mm",
        f"Humedad: {hoy_completo['humedad_relativa_pct']:.1f}%",
        f"Humedad Suelo: {hoy_completo['humedad_suelo_frac']:.2f}"
    ),
    specs=[[{}, {}], [{}, {}]]
)

# TEMPERATURA
fig3.add_trace(
    go.Indicator(
        mode="gauge+number",
        value=hoy_completo['temp_media_C'],
        title={'text': "°C"},
        gauge={'axis': {'range': [-10, 40]},
               'bar': {'color': '#FF6B6B'},
               'steps': [
                   {'range': [-10, 10], 'color': '#AED6F1'},
                   {'range': [10, 25], 'color': '#F9E79F'},
                   {'range': [25, 40], 'color': '#F5B7B1'}
               ]},
        domain={'x': [0, 0.45], 'y': [0.5, 1]}
    )
)

# LLUVIA
fig3.add_trace(
    go.Indicator(
        mode="gauge+number",
        value=hoy_completo['lluvia_mm'],
        title={'text': "mm"},
        gauge={'axis': {'range': [0, 100]},
               'bar': {'color': '#4A90E2'},
               'steps': [
                   {'range': [0, 5], 'color': '#E8F8F5'},
                   {'range': [5, 25], 'color': '#A9CCE3'},
                   {'range': [25, 100], 'color': '#2E86AB'}
               ]},
        domain={'x': [0.55, 1], 'y': [0.5, 1]}
    )
)

# HUMEDAD RELATIVA
fig3.add_trace(
    go.Indicator(
        mode="gauge+number",
        value=hoy_completo['humedad_relativa_pct'],
        title={'text': "%"},
        gauge={'axis': {'range': [0, 100]},
               'bar': {'color': '#2ECC71'},
               'steps': [
                   {'range': [0, 30], 'color': '#F5B7B1'},
                   {'range': [30, 70], 'color': '#F9E79F'},
                   {'range': [70, 100], 'color': '#A9DFBF'}
               ]},
        domain={'x': [0, 0.45], 'y': [0, 0.45]}
    )
)

# HUMEDAD SUELO
fig3.add_trace(
    go.Indicator(
        mode="gauge+number",
        value=hoy_completo['humedad_suelo_frac'] * 100,
        title={'text': "%"},
        gauge={'axis': {'range': [0, 100]},
               'bar': {'color': '#8B4513'},
               'steps': [
                   {'range': [0, 25], 'color': '#D7BDE2'},
                   {'range': [25, 75], 'color': '#F8B88B'},
                   {'range': [75, 100], 'color': '#82E0AA'}
               ]},
        domain={'x': [0.55, 1], 'y': [0, 0.45]}
    )
)

fig3.update_layout(
    title="Variables Climáticas de Hoy",
    height=600,
    width=1000,
    template='plotly_white'
)

fig3.show()
fig3.write_html("../reports/07_variables_hoy.html")
print("✓ Gráfica 3 guardada: 07_variables_hoy.html")

✓ Gráfica 3 guardada: 07_variables_hoy.html


## 6. COMPARACIÓN CON HISTÓRICO

**Muestra:** Hoy vs promedio histórico vs máximos

In [7]:
# GRÁFICA 4: COMPARACIÓN CON HISTÓRICO
comparacion_data = {
    'Variable': ['Temperatura (°C)', 'Lluvia (mm)', 'Humedad (%)', 'Humedad Suelo'],
    'Hoy': [
        hoy_completo['temp_media_C'],
        hoy_completo['lluvia_mm'],
        hoy_completo['humedad_relativa_pct'],
        hoy_completo['humedad_suelo_frac'] * 100
    ],
    'Promedio': [
        df['temp_media_C'].mean(),
        df['lluvia_mm'].mean(),
        df['humedad_relativa_pct'].mean(),
        df['humedad_suelo_frac'].mean() * 100
    ],
    'Máximo': [
        df['temp_media_C'].max(),
        df['lluvia_mm'].max(),
        df['humedad_relativa_pct'].max(),
        df['humedad_suelo_frac'].max() * 100
    ]
}

df_comp = pd.DataFrame(comparacion_data)

fig4 = go.Figure()

fig4.add_trace(go.Bar(
    name='Hoy',
    x=df_comp['Variable'],
    y=df_comp['Hoy'],
    marker_color='#E74C3C'
))

fig4.add_trace(go.Bar(
    name='Promedio Histórico',
    x=df_comp['Variable'],
    y=df_comp['Promedio'],
    marker_color='#95A5A6'
))

fig4.add_trace(go.Bar(
    name='Máximo Histórico',
    x=df_comp['Variable'],
    y=df_comp['Máximo'],
    marker_color='#BDC3C7'
))

fig4.update_layout(
    title='Comparación: Hoy vs Histórico',
    xaxis_title='Variable',
    yaxis_title='Valor',
    barmode='group',
    height=500,
    width=1000,
    template='plotly_white',
    hovermode='x unified'
)

fig4.show()
fig4.write_html("../reports/07_comparacion_historico.html")
print("✓ Gráfica 4 guardada: 07_comparacion_historico.html")

✓ Gráfica 4 guardada: 07_comparacion_historico.html


## 7. TENDENCIA ÚLTIMOS 30 DÍAS

**Muestra:** Evolución de la probabilidad de riesgo

In [8]:
# GRÁFICA 5: TENDENCIA ÚLTIMOS 30 DÍAS
df_ultimos_30 = df.tail(30).copy()

# Hacer predicciones para últimos 30 días
X_30 = df_ultimos_30[features].values
X_30_scaled = scaler.transform(X_30)
probabilidades_30 = modelo.predict_proba(X_30_scaled)[:, 1]

fig5 = go.Figure()

# Línea de probabilidad
fig5.add_trace(go.Scatter(
    x=df_ultimos_30.index,
    y=probabilidades_30 * 100,
    mode='lines+markers',
    name='Probabilidad de Riesgo',
    line=dict(color='#E74C3C', width=3),
    marker=dict(size=8),
    fill='tozeroy',
    fillcolor='rgba(231, 76, 60, 0.2)',
    hovertemplate='<b>%{x|%Y-%m-%d}</b><br>Probabilidad: %{y:.1f}%<extra></extra>'
))

# Línea de umbral (70%)
fig5.add_hline(y=70, line_dash="dash", line_color="red", 
              annotation_text="Umbral de Alerta (70%)",
              annotation_position="right")

# Línea de hoy
fig5.add_vline(x=ultima_fecha, line_dash="dot", line_color="blue",
              annotation_text="Hoy")

fig5.update_layout(
    title='Probabilidad de Riesgo - Últimos 30 Días',
    xaxis_title='Fecha',
    yaxis_title='Probabilidad (%)',
    height=500,
    width=1200,
    template='plotly_white',
    hovermode='x unified'
)

fig5.show()
fig5.write_html("../reports/07_tendencia_30_dias.html")
print("✓ Gráfica 5 guardada: 07_tendencia_30_dias.html")

TypeError: Addition/subtraction of integers and integer-arrays with Timestamp is no longer supported.  Instead of adding/subtracting `n`, use `n * obj.freq`

## 8. RESUMEN Y RECOMENDACIONES

**Análisis final y próximos pasos**

In [ ]:
print("\n" + "="*70)
print("RESUMEN DE PREDICCIÓN")
print("="*70)

print(f"\n📅 Fecha: {ultima_fecha.strftime('%A, %d de %B de %Y')}")
print(f"\n🎯 PREDICCIÓN: {'🔴 RIESGO DE CHAPULÍN' if probabilidad >= 0.5 else '🟢 SIN RIESGO'}")
print(f"   Probabilidad: {probabilidad*100:.1f}%")

print(f"\n📊 CONDICIONES CLIMÁTICAS:")
print(f"   • Temperatura: {hoy_completo['temp_media_C']:.1f}°C (promedio: {df['temp_media_C'].mean():.1f}°C)")
print(f"   • Lluvia: {hoy_completo['lluvia_mm']:.1f} mm (promedio: {df['lluvia_mm'].mean():.1f} mm)")
print(f"   • Humedad: {hoy_completo['humedad_relativa_pct']:.1f}% (promedio: {df['humedad_relativa_pct'].mean():.1f}%)")
print(f"   • Lluvia acumulada (14d): {hoy_completo['lluvia_14d']:.1f} mm")

print(f"\n💡 RECOMENDACIONES:")
if probabilidad >= 0.7:
    print("   🔴 ACCIÓN INMEDIATA REQUERIDA")
    print("   • Considerar aplicación de pesticida")
    print("   • Aumentar frecuencia de monitoreo a DIARIO")
    print("   • Notificar a agricultores locales")
elif probabilidad >= 0.5:
    print("   🟠 MONITOREO CERCANO")
    print("   • Aumentar vigilancia de campo")
    print("   • Revisar poblaciones de insectos")
    print("   • Estar listo para intervenir")
elif probabilidad >= 0.3:
    print("   🟡 MONITOREO ESTÁNDAR")
    print("   • Continuar vigilancia normal")
    print("   • Registrar cambios climáticos")
else:
    print("   🟢 BAJO RIESGO")
    print("   • Mantener vigilancia rutinaria")
    print("   • Continuar con prácticas normales")

print(f"\n📈 GRÁFICAS GENERADAS:")
print(f"   ✓ 07_gauge_riesgo_hoy.html")
print(f"   ✓ 07_alerta_hoy.html")
print(f"   ✓ 07_variables_hoy.html")
print(f"   ✓ 07_comparacion_historico.html")
print(f"   ✓ 07_tendencia_30_dias.html")

print(f"\n" + "="*70)
print("✅ PREDICCIÓN COMPLETADA")
print("="*70)